In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

TAMANIO_MUESTRA = 100_000

C:\Users\cance\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'],
        num_rows: 4500000
    })
    eval: Dataset({
        features: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'],
        num_rows: 350000
    })
})


In [3]:
print("Particiones disponibles:", list(dataset.keys()))

for particion, datos in dataset.items():
    print(f"{particion}: {datos.num_rows:,} filas y {datos.num_columns} columnas")

particion_trabajo = "train" if "train" in dataset else list(dataset.keys())[0]
datos_completos = dataset[particion_trabajo]

print("\nPartición seleccionada:", particion_trabajo)
print("Columnas:", datos_completos.column_names)

Particiones disponibles: ['train', 'eval']
train: 4,500,000 filas y 19 columnas
eval: 350,000 filas y 19 columnas

Partición seleccionada: train
Columnas: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level']


In [4]:
n_muestra = min(TAMANIO_MUESTRA, datos_completos.num_rows)

datos_muestra = datos_completos.select(range(n_muestra))

df = datos_muestra.to_pandas().reset_index(drop=True)

print(f"Registros originales: {datos_completos.num_rows:,}")
print(f"Registros seleccionados en secuencia: {len(df):,}")
print(f"Columnas: {df.shape[1]}")
display(df.head())

Registros originales: 4,500,000
Registros seleccionados en secuencia: 100,000
Columnas: 19


,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,dt,sale_amount,hours_sale,stock_hour6_22_cnt,hours_stock_status,discount,holiday_flag,activity_flag,precpt,avg_temperature,avg_humidity,avg_wind_level
0,0,0,0,5,6,65,38,2024-03-28,0.1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, ...",0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0,0,0,1.6999,15.48,73.54,1.97
1,0,0,0,5,6,65,38,2024-03-29,0.1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, ...",1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0,0,0,3.0190,15.08,76.56,1.71
2,0,0,0,5,6,65,38,2024-03-30,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0,1,0,2.0942,15.91,76.47,1.73
3,0,0,0,5,6,65,38,2024-03-31,0.1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, ...",11,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, ...",1.0,1,0,1.5618,16.13,77.40,1.76
4,0,0,0,5,6,65,38,2024-04-01,0.2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, ...",8,"[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...",1.0,0,0,3.5386,15.37,78.26,1.25


In [5]:
def crear_etiquetas(serie, prefijo, ancho=4):
    valores = sorted(serie.dropna().unique(), key=str)
    return {
        valor: f"{prefijo} {numero:0{ancho}d}"
        for numero, valor in enumerate(valores, start=1)
    }

mapa_productos = crear_etiquetas(df["product_id"], "Producto", ancho=4)
df["nombre_producto"] = df["product_id"].map(mapa_productos)

if "store_id" in df.columns:
    mapa_tiendas = crear_etiquetas(df["store_id"], "Tienda", ancho=3)
    df["nombre_tienda"] = df["store_id"].map(mapa_tiendas)

columnas_categoria = [
    columna for columna in [
        "first_category_id",
        "second_category_id",
        "third_category_id",
    ]
    if columna in df.columns
]

for nivel, columna in enumerate(columnas_categoria, start=1):
    mapa_categoria = crear_etiquetas(
        df[columna], f"Categoría N{nivel}", ancho=3
    )
    df[f"nombre_categoria_n{nivel}"] = df[columna].map(mapa_categoria)

print(f"Productos diferentes en la muestra: {df['product_id'].nunique():,}")
if "store_id" in df.columns:
    print(f"Tiendas diferentes en la muestra: {df['store_id'].nunique():,}")

columnas_vista = [
    columna for columna in [
        "product_id",
        "nombre_producto",
        "store_id",
        "nombre_tienda",
        "first_category_id",
        "nombre_categoria_n1",
        "second_category_id",
        "nombre_categoria_n2",
        "third_category_id",
        "nombre_categoria_n3",
    ]
    if columna in df.columns
]

display(df[columnas_vista].drop_duplicates().head(15))

Productos diferentes en la muestra: 273
Tiendas diferentes en la muestra: 13


,product_id,nombre_producto,store_id,nombre_tienda,first_category_id,nombre_categoria_n1,second_category_id,nombre_categoria_n2,third_category_id,nombre_categoria_n3
0,38,Producto 0105,0,Tienda 001,5,Categoría N1 023,6,Categoría N2 040,65,Categoría N3 122
90,834,Producto 0256,0,Tienda 001,28,Categoría N1 017,72,Categoría N2 052,154,Categoría N3 039
180,411,Producto 0112,0,Tienda 001,28,Categoría N1 017,72,Categoría N2 052,218,Categoría N3 085
270,686,Producto 0200,0,Tienda 001,0,Categoría N1 001,21,Categoría N2 011,221,Categoría N3 087
360,580,Producto 0158,0,Tienda 001,29,Categoría N1 018,76,Categoría N2 056,60,Categoría N3 119
450,596,Producto 0160,0,Tienda 001,29,Categoría N1 018,76,Categoría N2 056,60,Categoría N3 119
540,740,Producto 0221,0,Tienda 001,29,Categoría N1 018,76,Categoría N2 056,60,Categoría N3 119
630,379,Producto 0104,0,Tienda 001,29,Categoría N1 018,76,Categoría N2 056,231,Categoría N3 096
720,4,Producto 0109,0,Tienda 001,29,Categoría N1 018,78,Categoría N2 058,82,Categoría N3 137
810,600,Producto 0164,0,Tienda 001,29,Categoría N1 018,78,Categoría N2 058,157,Categoría N3 041


In [6]:
columnas_catalogo = [
    columna for columna in [
        "product_id",
        "nombre_producto",
        "first_category_id",
        "nombre_categoria_n1",
        "second_category_id",
        "nombre_categoria_n2",
        "third_category_id",
        "nombre_categoria_n3",
    ]
    if columna in df.columns
]

catalogo_productos = (
    df[columnas_catalogo]
    .drop_duplicates(subset=["product_id"])
    .sort_values("nombre_producto")
    .reset_index(drop=True)
)

display(catalogo_productos.head(20))

,product_id,nombre_producto,first_category_id,nombre_categoria_n1,second_category_id,nombre_categoria_n2,third_category_id,nombre_categoria_n3
0,100,Producto 0001,11,Categoría N1 004,60,Categoría N2 041,15,Categoría N3 037
1,104,Producto 0002,20,Categoría N1 010,50,Categoría N2 032,2,Categoría N3 075
2,108,Producto 0003,8,Categoría N1 025,29,Categoría N2 017,113,Categoría N3 015
3,109,Producto 0004,8,Categoría N1 025,29,Categoría N2 017,113,Categoría N3 015
4,11,Producto 0005,18,Categoría N1 009,80,Categoría N2 061,86,Categoría N3 139
5,110,Producto 0006,31,Categoría N1 021,79,Categoría N2 059,121,Categoría N3 020
6,114,Producto 0007,10,Categoría N1 003,33,Categoría N2 020,72,Categoría N3 128
7,115,Producto 0008,10,Categoría N1 003,33,Categoría N2 020,181,Categoría N3 063
8,116,Producto 0009,10,Categoría N1 003,33,Categoría N2 020,177,Categoría N3 058
9,117,Producto 0010,4,Categoría N1 022,28,Categoría N2 016,1,Categoría N3 002


In [ ]:
ruta_salida = "../../data/freshretailnet_muestra_100k_secuencial.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8")

print(f"Muestra guardada correctamente en: {ruta_salida}")
print(f"Filas guardadas: {len(df):,}")
print(f"Columnas guardadas: {df.shape[1]}")

Muestra guardada correctamente en: ../data/freshretailnet_muestra_100k_secuencial.csv
Filas guardadas: 100,000
Columnas guardadas: 24


In [8]:
print("Dimensiones finales:", df.shape)
print("\nTipos de datos:")
display(df.dtypes.to_frame("tipo"))

print("\nValores faltantes:")
faltantes = df.isna().sum().sort_values(ascending=False)
display(faltantes[faltantes > 0].to_frame("cantidad"))

print("\nFilas duplicadas:", df.duplicated().sum())
display(df.head())

Dimensiones finales: (100000, 24)

Tipos de datos:


,tipo
city_id,int64
store_id,int64
management_group_id,int64
first_category_id,int64
second_category_id,int64
third_category_id,int64
product_id,int64
dt,str
sale_amount,float64
hours_sale,object



Valores faltantes:


,cantidad


TypeError: unhashable type: 'numpy.ndarray'